In [ ]:
from collections import defaultdict, Counter, OrderedDict
from sklearn.preprocessing import MinMaxScaler

import pandas as pd
import numpy as np
import glob
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pickle

from scipy import stats
from toolz.functoolz import compose
from pprint import pprint
from river.drift import ADWIN
import matplotlib.pyplot as plt

from notion.client import NotionClient
from notion.block import  ImageBlock

import plotly.graph_objects as go

from tqdm.notebook import tqdm

from multiprocessing import Pool
from datetime import datetime

# Data preparation

In [ ]:
paths = glob.glob("data/log_count/Affluences Logfrequency Data/*.csv")
dfs = [pd.read_csv(path, parse_dates=["timestamp"], usecols=["timestamp", "n_message"]) for path in paths]

path_cluster_dict = {
    p.replace(".png", "").split("/")[-1]: int(p.split("/")[-2])
    for p in glob.glob("data/clusters/labels/*/*.png")
}

dfs_clusters = list(
    map(
        lambda x: path_cluster_dict.get(
            x.replace("data/log_count/Affluences Logfrequency Data/", "").replace(
                ".csv", ""
            )
        ),
        paths,
    )
)

In [ ]:
def scale_df(df):
    result = df.copy()
    result["n_message"] = MinMaxScaler().fit_transform(result.n_message.values.reshape(-1, 1))
    return result

def remove_outliers(df):
    z_scores = stats.zscore(df.n_message)
    abs_z_scores = np.abs(z_scores)
    filtered_entries = abs_z_scores < 3
    new_df = df[filtered_entries]
    if len(new_df) > 10:
        return new_df
    else:
        return df

def trim(df):
    temp = df.copy()
    q = np.quantile(temp.n_message, .99)
    mq = np.quantile(temp.n_message, .01)
#     return temp[(temp.n_message < q) & (temp.n_message > mq)]
    return temp[temp.n_message < q]

def generate_features(df):
#     print(df.shape)
    mean = df["n_message"].mean()
    std = df["n_message"].std()
    unique_vals_count = len(df["n_message"].value_counts())
    median = df["n_message"].median()
    more_than_quantile95 = (
        df[df["n_message"] > df["n_message"].quantile(0.95)].shape[0] / df.shape[0]
    )
    fraction_of_zeros = df[df["n_message"] == 0].shape[0] / df.shape[0]
    fraction_of_nonzeros = df[df["n_message"] != 0].shape[0] / df.shape[0]
    ordered_counts = OrderedDict(
        sorted(Counter(df["n_message"]).items(), key=lambda x: x[1], reverse=True)
    )
    perc_val = {v: c / len(dfs[0]) for v, c in ordered_counts.items()}
    # with counter features
    cumsum_perc = np.cumsum(list(ordered_counts.values())) / len(df)
    cumsum_perc_top_1 = cumsum_perc[0] if len(cumsum_perc) >= 1 else 1.0
    cumsum_perc_top_2 = cumsum_perc[1] if len(cumsum_perc) >= 2 else 1.0
    cumsum_perc_top_3 = cumsum_perc[2] if len(cumsum_perc) >= 3 else 1.0
    cumsum_perc_top_4 = cumsum_perc[3] if len(cumsum_perc) >= 4 else 1.0
    cumsum_perc_top_5 = cumsum_perc[4] if len(cumsum_perc) >= 5 else 1.0
    min_value = df["n_message"].min()
    min_value_perc = perc_val[min_value]
    max_value = df["n_message"].max()
    max_value_perc = perc_val[max_value]
    total_values = len(df)
    total_zeros = df[df["n_message"] == 0].shape[0]
    total_nonzeros = total_values - total_zeros
    nonzero_mean = df[df["n_message"] != 0]["n_message"].mean() if len(df[df["n_message"] != 0]) > 3 else 0
    nonzero_std = df[df["n_message"] != 0]["n_message"].std() if len(df[df["n_message"] != 0]) > 3 else 0
    return pd.DataFrame(
        {
            "mean": [mean],
            "std": [std],
            "unique_vals_count": [unique_vals_count],
            "median": [median],
            "more_than_quantile95": [more_than_quantile95],
            "fraction_of_zeros": [fraction_of_zeros],
            "fraction_of_nonzeros": [fraction_of_nonzeros],
            "cumsum_perc_top_1": [cumsum_perc_top_1],
            "cumsum_perc_top_2": [cumsum_perc_top_2],
            "cumsum_perc_top_3": [cumsum_perc_top_3],
            "cumsum_perc_top_4": [cumsum_perc_top_4],
            "cumsum_perc_top_5": [cumsum_perc_top_5],
            "total_values": [total_values],
            "total_zeros": [total_zeros],
            "total_nonzeros": [total_nonzeros],
            "nonzero_mean": [nonzero_mean],
            "nonzero_std": [nonzero_std],
        }
    )

def split_into_buckets(df, n_buckets=5):
    step = len(df) // n_buckets
    res = [df[i*step: (i+1)*step] for i in range(n_buckets)]
    return res

def generate_classification_features(df, n=20):
    # features for initial set
    general_features = generate_features(df)
    # features for diff set
    diff_general_features = generate_features(df.diff().dropna())
    diff_general_features.columns = [
        "diff_" + col for col in diff_general_features.columns
    ]
    # bucket split
    df2 = pd.concat(map(generate_features, split_into_buckets(df, n_buckets=n)))
    # mean bucket
    mean_bucket = pd.DataFrame(df2.mean()).T
    mean_bucket.columns = ["mean_bucket_" + col for col in mean_bucket.columns]
    # std bucket
    std_bucket = pd.DataFrame(df2.std()).T
    std_bucket.columns = ["std_bucket_" + col for col in std_bucket.columns]
    # bucket split for diff
    df3 = pd.concat(
        map(generate_features, split_into_buckets(df.diff().dropna(), n_buckets=n))
    )
    # mean bucket
    mean_bucket_diff = pd.DataFrame(df3.mean()).T
    mean_bucket_diff.columns = [
        "mean_bucket_diff_" + col for col in mean_bucket_diff.columns
    ]
    # std bucket
    std_bucket_diff = pd.DataFrame(df3.std()).T
    std_bucket_diff.columns = [
        "std_bucket_diff_" + col for col in std_bucket_diff.columns
    ]
    # result
    result = pd.concat(
        [
            general_features,
            mean_bucket,
            std_bucket,
            mean_bucket_diff,
            std_bucket_diff,
            diff_general_features,
        ],
        axis=1,
    )
    return result

def smooth(df):
    return df.set_index("timestamp").rolling("1H").mean().dropna()

def remove_drift(df):
    adwin = ADWIN()

    dates = []
    for time, value in df.set_index("timestamp").iterrows():
        val = value["n_message"]
        in_drift, in_warning = adwin.update(val)
        if in_drift:
            dates.append(time)
    if not dates:
        return df
    slices = []
    cnt_slice = 0
    start_ts = None
    while dates:
        end_ts = dates.pop(0)
        if start_ts is None:
            slices.append(df.set_index("timestamp")[:end_ts])
        else:
            avg = slices[-1].mean().values[0]
            temp_df = df.set_index("timestamp")[start_ts:end_ts]
            delta = avg - temp_df.mean().values[0]
            slices.append(temp_df - abs(delta) if delta < 0 else temp_df + abs(delta))
        start_ts = end_ts
    avg = slices[-1].mean().values[0]
    temp_df = df.set_index("timestamp")[end_ts:]
    delta = avg - temp_df.mean().values[0]
    slices.append(temp_df - abs(delta) if delta < 0 else temp_df + abs(delta))
    return pd.concat(slices).reset_index()

In [ ]:
def point_in_interval(point, interval):
    return point >= interval[0] and point <= interval[1]


def count_point_in_interval(points, interval):
    return sum([1.0 if point_in_interval(point, interval) else 0.0 for point in points])


def max_intersection(l, delta=5 * 60, min_outliers=10):
    output = [[(t - delta, "begin"), (t + delta, "end")] for t in l]
    cnt = 0

    sorted_intervals = sorted(
        set([i for sublist in output for i in sublist]), key=lambda x: (x[0], x[1])
    )
    spans = []
    last_start, last_end = 0, 0
    for d in sorted_intervals:
        if d[1] == "end":
            if cnt != 0:
                cnt -= 1
            if cnt == 0:
                spans.append([last_start, d[0]])
        else:
            if cnt == 0:
                last_start = d[0]
            cnt += 1

    span_counts = sorted(
        [(span, count_point_in_interval(l, span)) for span in spans],
        key=lambda x: x[1],
        reverse=True,
    )
    #     print(span_counts)
    if span_counts and span_counts[0][1] > min_outliers:
        return span_counts[0][0]
    else:
        return []


def remove_outliers2(data):
    df = data.copy()
    z_scores = stats.zscore(df.n_message)
    abs_z_scores = np.abs(z_scores)
    filtered_entries = abs_z_scores < 3
    df["keep"] = filtered_entries
    outliers = abs_z_scores >= 3
    outliers_diff = df[outliers].diff().dropna()["timestamp"].dt.seconds.values
    timespan_days = (df.timestamp.max() - df.timestamp.min()).days
    max_interval = max_intersection(outliers_diff, min_outliers=timespan_days)
    if max_interval:
        points_in = [False] + [
            point_in_interval(point, max_interval) for point in outliers_diff
        ]
        potetntial_idx_keep = df[outliers].diff()[points_in].index
        potential_keep = df.loc[potetntial_idx_keep]
        median = np.median(potential_keep.n_message)
        filtered_entries = potential_keep.n_message.values < 1.5 * median
        idx_keep = potential_keep[filtered_entries].index
        df.at[idx_keep, "keep"] = True
    new_df = df[df["keep"]][["timestamp", "n_message"]]
    if len(new_df) > 10:
        return new_df
    else:
        return df


def make_directory(directory):
    """
    Create a directory
    :param directory: path to the folder to create
    :type directory: str
    :return: None
    :rtype: object
    """
    if not (os.path.exists(directory)):
        os.makedirs(directory)
        print(f"Created a directory: '{directory}'")

In [ ]:
process_scale = compose(generate_classification_features, scale_df)
process_outliers = compose(generate_classification_features, remove_outliers)
process_smooth_outliers = compose(generate_classification_features, smooth, remove_outliers)
process_smooth_outliers_drift = compose(generate_classification_features, smooth, remove_outliers, remove_drift)
process_outliers_drift = compose(generate_classification_features, remove_outliers, remove_drift)
process_outliers2_drift = compose(generate_classification_features, remove_outliers2, remove_drift)
outliers_drift = compose(remove_outliers, remove_drift)

In [ ]:
cluster_description = {
    0: "usually the same values observed",
    1: "dense data around the mean value",
    2: "random noisy data",
    3: "sparse data with no patterns",
    4: "data with seasonal patterns",
}

In [ ]:
result = []
count = 0
for dfs_list, cluster in zip(map(split_into_buckets, dfs), dfs_clusters):
    for df in tqdm(dfs_list):
        temp = process_outliers2_drift(df) 
        temp["cluster"] = cluster
        result.append(temp)
    count += 1
    print("Processed", count, "datasets")

In [ ]:
X = pd.concat(result)

# Classifier

In [ ]:
X.shape

In [ ]:
cluster_description

In [ ]:
y_RF = X.cluster.values
X_RF = X[[col for col in X.columns if col != "cluster"]].values
X_train_RF, X_test_RF, y_train_RF, y_test_RF = train_test_split(
    X_RF, y_RF, test_size=0.33, random_state=42, stratify=y_RF
)
# define the model
model = RandomForestClassifier()
# fit the model
model.fit(X_train_RF, y_train_RF)

In [ ]:
confusion_matrix(y_true=y_test_RF, y_pred=model.predict(X_test_RF))

In [ ]:
print(classification_report(y_true=y_test_RF, y_pred=model.predict(X_test_RF)))

In [ ]:
sorted(
    list(
        zip(
            X[[col for col in X.columns if col != "cluster"]].columns,
            model.feature_importances_,
        )
    ),
    key=lambda x: x[1], reverse=True
)[:10]

In [ ]:
Counter(y_train_RF)

In [ ]:
for cluster, count in Counter(y_train_RF).items():
    print("Cluster", cluster, '(', cluster_description[cluster], ") :", count, "datasets")

In [ ]:
# X_features.to_csv("data/temp_files/X_features.csv")

# # save the model to disk
# filename = "data/temp_files/RF_model.sav"
# pickle.dump(model, open(filename, "wb"))

# pd.to_pickle(idx_cluster_dict, "data/temp_files/idx_cluster_dict.pickle")

# # load the model from disk
# loaded_model = pickle.load(open(filename, 'rb'))
# print(loaded_model.score(X_test_RF, y_test_RF))

# Evaluation

In [ ]:
# path = "zg_data"
path = "evaluation"

paths_zg = glob.glob(f"data/{path}/*.csv")
dfs_zg = [
    pd.read_csv(path, parse_dates=["timestamp"], usecols=["timestamp", "n_message"])
    for path in paths_zg
]

if path == "zg_data":
    X_eval = pd.concat(map(process_outliers2_drift, dfs_zg))
else:
    with Pool() as P:
        res = P.map(process_outliers2_drift, dfs_zg)
    X_eval = pd.concat(res)

zg_clusters = model.predict(X_eval.values)

zg_proba = model.predict_proba(X_eval.values)

### Save predicted proba

In [ ]:
# res_tmp = []
# for topic, probs in zip(map(lambda x: x.replace(".csv", "").split("/")[-1], paths_zg), zg_proba):
#     d = {"topic": topic}
#     for idx, proba in enumerate(probs):
#         d.update({cluster_description[idx]: probs[idx]})
#     res_tmp.append(d)

# pd.DataFrame(res_tmp).to_csv(f"results/topic_proba_{str(datetime.now())}.csv", index=False)

### Score

In [ ]:
proba_df = pd.read_csv(
    "results/topic_proba_2021-09-21 11:16:22.554513.csv", index_col=["topic"]
)
proba_dict = proba_df.to_dict(orient="index")

description_cluster = {v: k for k, v in cluster_description.items()}

topic_description_label_pred = proba_df.idxmax(axis=1).to_dict()
topic_cluster_label_pred = {
    topic: description_cluster[description]
    for topic, description in topic_description_label_pred.items()
}

predicted_cluster = {}
score, cnt = 0, 0
true_classes = []
predicted_classes = []
same_values_classified_as_random = []
for cluster in range(4):
    for path in glob.glob(f"results/evaluation_labels/{cluster}/*.png"):
        topic = path.replace("_evaluation.png", "").split("/")[-1]
        score += proba_dict[topic][cluster_description[cluster]]
        cnt += 1
        true_classes.append(cluster)
        predicted_classes.append(topic_cluster_label_pred[topic])
        if cluster == 0 and topic_cluster_label_pred[topic] == 2:
            same_values_classified_as_random.append(topic)
print("Accuracy:", score / cnt)

In [ ]:
cluster_description

In [ ]:
confusion_matrix(y_true=true_classes, y_pred=predicted_classes)

## Save evaluation charts

In [ ]:
def plot_different_transformations(df):
    smooth_remove_outliers = compose(smooth, remove_outliers)
    fig, axs = plt.subplots(3, 1, figsize=(12, 10))
    df.set_index("timestamp").plot(ax=axs[0], title='Initial ts')
    remove_outliers(df).set_index("timestamp").plot(ax=axs[1], title="remove outliers")
    smooth_remove_outliers(df).plot(ax=axs[2], title="smooth + remove outliers")
    fig.tight_layout()

In [ ]:
TOKEN_V2 = "928114611bccc05e601f9ef2bdbe48a246b6edc7669f803d043295f804925ce70bb65ccdee42d88dc9641c85753d232bf621ed9a5cab5e4859e416b3928de063b76b02f2a06d1ff1faec1c56026d"
# URL_TO_PAGE = (
#     "https://www.notion.so/packetai/10-09-2021-2ee8d10870934fda858f55875fdffd0a"
# )
URL_TO_PAGE = (
    "https://www.notion.so/packetai/919cecd1cc8c417c9acf47140158df0f"
)
client = NotionClient(token_v2=TOKEN_V2)
page = client.get_block(URL_TO_PAGE)

In [ ]:
folder_name = "_".join(["evaluation", str(datetime.now())])
make_directory(os.path.join("results", folder_name))
for i in range(5):
    make_directory(os.path.join("results", folder_name, str(i)))
for idx, c in enumerate(tqdm(zg_clusters)):
    df = dfs_zg[idx].set_index("timestamp")
    proba = zg_proba[idx]
    proba_str = "\n".join(
        [f"{cluster_description[i]} : {j}" for i, j in zip(range(len(proba)), proba)]
    )
    name = paths_zg[idx].split("/")[-1].replace(".csv", "_evaluation")
    ax = df.plot(
        figsize=(15, 4),
        title=f"Predicted cluster {c}: {cluster_description[c]}\nProbabilities for other clusters:\n{proba_str}\n"
        + name,
    )
    ax.get_figure().savefig(
        f"results/{folder_name}/{c}/" + name + ".png", bbox_inches="tight"
    )
    plt.close()
    # Save to Notion
#     image_block = page.children.add_new(ImageBlock)
#     image_block.upload_file("results/evaluation/" + name + ".png",)

## Check data by query

In [ ]:
[i for i in paths_zg if "nodeportal" in i]

In [ ]:
query = "6040a96cdcc614001153fc55-docker-log-nodeportal_portal-stashed"
# query = same_values_classified_as_random[6]
names = [
    x.replace(".csv", "").replace("-stashed", "").split("log-")[-1] for x in paths_zg
]
name_df_dict = dict(zip(names, dfs_zg))
selected_key = [key for key in name_df_dict.keys() if key in query][0]
df = name_df_dict[selected_key]
# # Create traces
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.timestamp, y=df.n_message, mode="lines", name="lines"))
fig.update_layout(
    title=selected_key, xaxis_title="Date", yaxis_title="Log count",
)
fig.show()

## Plot all data

In [ ]:
names = [x.replace('.csv', '').replace('-stashed', '').split('log-')[-1] for x in paths_zg]
visible_d = {}
for idx, name in enumerate(names):
    d = [False] * len(names)
    d[idx] = True
    visible_d[name] = d

# Create traces
fig = go.Figure()
for df in dfs_zg:
    fig.add_trace(
        go.Scatter(x=df.timestamp, y=df.n_message, mode="lines", name="lines")
    )


fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=list(
                [
                    dict(
                        label=name,
                        method="update",
                        args=[{"visible": visible_d[name]}, {"title": name},],
                    )
                    for name in names
                ]
            ),
        )
    ]
)

fig.show()